# 광파오븐 × 신혼부부 — 네이버 지식인 크롤링
- **목적**: 신혼부부의 광파오븐 Pain Point 및 니즈 수집 (가설별 키워드 분류)
- **수집 항목**: 제목 / 질문 본문 / 채택 답변 / 키워드 카테고리
- **실행 환경**: Google Colab 권장


In [ ]:
# Google Colab 실행 시 아래 두 줄 주석 해제
# !apt-get install -y default-jdk > /dev/null 2>&1
# !pip install selenium beautifulsoup4 pandas tqdm -q

!pip install selenium beautifulsoup4 pandas tqdm -q


In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup as bs
from tqdm import tqdm
import time, re, random
import pandas as pd

# =====================================================================
# 키워드 정의 — 신혼부부 맥락 가설별 분류
# =====================================================================

KEYWORDS = {

    # ── 가설 A: 혼수·입주 시점 구매 탐색
    # 결혼 준비 or 신혼집 입주 전후로 가전을 새로 세팅하는 맥락
    # → "언제, 어떤 기준으로 광파오븐을 선택하는가" 파악
    "A_혼수입주": [
        "신혼 광파오븐",
        "혼수 광파오븐",
        "신혼집 오븐 추천",
        "신혼 가전 광파오븐",
        "혼수 가전 오븐",
        "신혼집 주방가전 광파오븐",
    ],

    # ── 가설 B: 경쟁 가전 비교 (구매 장벽)
    # 에어프라이어·전자레인지·오븐레인지와의 비교 고민
    # → 광파오븐을 선택하지 않는 이유 / 선택하는 이유 파악
    "B_가전비교": [
        "신혼 에어프라이어 광파오븐",
        "신혼 전자레인지 광파오븐",
        "신혼 오븐레인지 광파오븐",
        "신혼부부 에어프라이어 오븐 뭐 살까",
        "신혼 주방가전 뭐 사야",
        "신혼 가전 에어프라이어 오븐 추천",
    ],

    # ── 가설 C: 요리 시작 니즈 (첫 살림)
    # 결혼 후 처음 본격적으로 요리를 시작하는 맥락
    # → 사용 편의성·레시피 활용에 대한 니즈 파악
    "C_요리시작": [
        "신혼 오븐 요리",
        "신혼부부 오븐 사용법",
        "신혼 광파오븐 레시피",
        "신혼집 요리 오븐",
        "둘이 사는데 광파오븐",
        "맞벌이 신혼 광파오븐",
    ],

    # ── 가설 D: 건강·식단 관리 니즈
    # 결혼 후 배우자 건강을 챙기기 시작하는 맥락
    # → 저염·건강 조리 도구로서 광파오븐에 대한 기대
    "D_건강식단": [
        "신혼 건강 요리 오븐",
        "신혼 다이어트 광파오븐",
        "신혼 저염식 오븐",
        "남편 건강 요리 광파오븐",
        "아내 건강식 오븐",
    ],

    # ── 가설 E: 공간·인테리어 적합성
    # 신혼집 주방 크기·인테리어와 가전이 어울리는지 고민하는 맥락
    # → 크기·디자인 Pain Point 파악
    "E_공간인테리어": [
        "신혼집 주방 좁은데 광파오븐",
        "아파트 신혼 광파오븐",
        "신혼집 빌트인 오븐",
        "신혼집 오븐 크기",
        "원베드 신혼 광파오븐",
    ],

    # ── 가설 F: 임신·출산 이후 전환 니즈
    # 신혼에서 육아 초기로 넘어가는 시점의 조리 니즈 변화
    # → 이유식·아기 간식 조리 맥락 포착
    "F_육아전환": [
        "신혼 임신 광파오븐",
        "임신 중 오븐 요리",
        "아기 이유식 광파오븐",
        "출산 후 오븐 활용",
        "아기 간식 광파오븐",
    ],
}

# 키워드 플랫하게 펼치기
keyword_list = []
for cat, kws in KEYWORDS.items():
    for kw in kws:
        keyword_list.append({"category": cat, "keyword": kw})

print(f"총 키워드: {len(keyword_list)}개")
for item in keyword_list:
    print(f"  [{item['category']}] {item['keyword']}")


In [ ]:
def get_driver():
    options = Options()
    options.add_argument("--headless")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--disable-gpu")
    options.add_argument("--window-size=1920,1080")
    options.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0.0.0 Safari/537.36"
    )
    driver = webdriver.Chrome(options=options)
    return driver

driver = get_driver()
print("드라이버 실행 완료")


In [ ]:
def get_kin_urls(driver, keyword, max_pages=5):
    """
    네이버 지식인 키워드 검색 → 질문 URL 목록 반환
    max_pages: 수집할 페이지 수 (1페이지 ≈ 10건)
    """
    urls = []
    base_url = "https://kin.naver.com/search/list.naver"

    for page in range(1, max_pages + 1):
        start = (page - 1) * 10 + 1
        search_url = f"{base_url}?query={keyword}&page={page}&startRank={start}"

        try:
            driver.get(search_url)
            time.sleep(random.uniform(1.5, 2.5))

            soup = bs(driver.page_source, "html.parser")
            items = soup.select("ul.basic1 li dl dt a")

            if not items:
                print(f"  [{keyword}] p{page} — 결과 없음, 종료")
                break

            for a in items:
                href = a.get("href", "")
                if href and "detail.naver" in href:
                    full_url = "https://kin.naver.com" + href if href.startswith("/") else href
                    urls.append(full_url)

            print(f"  [{keyword}] p{page} — {len(items)}건 수집 (누적: {len(urls)})")

        except Exception as e:
            print(f"  [{keyword}] p{page} — 오류: {e}")
            continue

    return list(set(urls))


In [ ]:
def get_kin_detail(driver, url):
    """
    지식인 질문 상세 페이지 → 제목 / 본문 / 채택답변 추출
    """
    try:
        driver.get(url)
        time.sleep(random.uniform(1.5, 2.5))
        soup = bs(driver.page_source, "html.parser")

        # 제목
        title_tag = (
            soup.select_one("h3.title")
            or soup.select_one("div.title")
            or soup.select_one(".question-title")
        )
        title = title_tag.get_text(strip=True) if title_tag else ""

        # 질문 본문
        content_tag = (
            soup.select_one("div.c-heading__content")
            or soup.select_one("div.questionDetail")
            or soup.select_one("div#questionDetail")
            or soup.select_one("div.question_content")
        )
        content = content_tag.get_text(strip=True) if content_tag else ""

        # 채택 답변
        answer_tag = (
            soup.select_one("div.answer-content")
            or soup.select_one("div.se-main-container")
            or soup.select_one("div#answerDetail")
            or soup.select_one("div.answerDetail")
        )
        answer = answer_tag.get_text(strip=True) if answer_tag else ""

        def clean(text):
            return re.sub(r"\s+", " ", text).strip()

        return {
            "title":   clean(title),
            "content": clean(content),
            "answer":  clean(answer),
            "url":     url,
        }

    except Exception as e:
        return {"title": "", "content": "", "answer": str(e), "url": url}


In [ ]:
# =====================================================================
# 메인 크롤링 실행
# =====================================================================
# MAX_PAGES: 키워드당 최대 페이지 수
# 키워드 34개 × 5페이지 × 10건 = 최대 1,700건 (중복 제거 후 600~900건 예상)

MAX_PAGES = 5  # 필요 시 조정

all_data = []
seen_urls = set()

for item in tqdm(keyword_list, desc="전체 키워드 진행"):
    category = item["category"]
    keyword  = item["keyword"]

    print(f"\n[{keyword}] URL 수집 시작")
    urls = get_kin_urls(driver, keyword, max_pages=MAX_PAGES)

    new_urls = [u for u in urls if u not in seen_urls]
    seen_urls.update(new_urls)
    print(f"  → 신규 URL: {len(new_urls)}개 (중복 제외)")

    for url in tqdm(new_urls, desc=f"  상세 수집 [{keyword}]", leave=False):
        detail = get_kin_detail(driver, url)
        detail["keyword"]  = keyword
        detail["category"] = category
        all_data.append(detail)
        time.sleep(random.uniform(0.8, 1.5))

print(f"\n✅ 전체 수집 완료 — {len(all_data)}건")


In [ ]:
df = pd.DataFrame(all_data)
df = df[["category", "keyword", "title", "content", "answer", "url"]]
df = df[(df["title"] != "") | (df["content"] != "")]
df = df.reset_index(drop=True)

print(f"유효 데이터: {len(df)}건")
print("\n카테고리별 건수:")
print(df["category"].value_counts())

output_path = "광파오븐_신혼부부_지식인_크롤링결과.csv"
df.to_csv(output_path, index=False, encoding="utf-8-sig")
print(f"\n✅ 저장 완료: {output_path}")
df.head(3)


In [ ]:
driver.quit()
print("드라이버 종료 완료")


In [ ]:
# 가설별 대표 질문 샘플 확인
print("=" * 60)
for cat in df["category"].unique():
    subset = df[df["category"] == cat]
    print(f"\n[{cat}] — {len(subset)}건")
    for _, row in subset.head(2).iterrows():
        print(f"  Q: {row['title'][:60]}...")
        if row["answer"]:
            print(f"  A: {row['answer'][:80]}...")
    print()
